## Cell 1: Mount Google Drive & Check GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU DETECTED - Go to Runtime → Change runtime type → Select GPU')

## Cell 2: Install Dependencies

In [ ]:
# Install PyTorch with CUDA support
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install -q fastapi uvicorn python-multipart Pillow numpy scikit-learn opencv-python-headless httpx pydantic matplotlib python-dotenv scipy tqdm

print('✓ All dependencies installed!')

## Cell 3: Navigate to Project & Check Dataset

In [ ]:
import os
import sys

# Navigate to project - ADJUST PATH IF NEEDED
project_path = '/content/drive/MyDrive/medical-ai'
backend_path = os.path.join(project_path, 'backend')

if not os.path.exists(project_path):
    print(f'ERROR: {project_path} not found!')
    print('Make sure you uploaded the medical-ai folder to Google Drive')
else:
    print(f'✓ Found project at: {project_path}')
    os.chdir(backend_path)
    sys.path.insert(0, backend_path)
    
    # Check dataset
    train_path = './data/rickets/train'
    if os.path.exists(train_path):
        print(f'\n✓ Dataset found in {train_path}')
        for cls in sorted(os.listdir(train_path)):
            cls_path = os.path.join(train_path, cls)
            count = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
            print(f'  - {cls}: {count} images')
    else:
        print(f'⚠️  Dataset NOT found at {train_path}')
        print('   Will prepare from raw data...')

## Cell 4: Prepare Dataset (if needed)

In [ ]:
# Only run if dataset is not prepared
import os

train_path = './data/rickets/train'
if not os.path.exists(train_path):
    print('Preparing Rickets dataset...')
    os.chdir('/content/drive/MyDrive/medical-ai')
    !python backend/prepare_data.py --dataset rickets --src Dataset/Rickets --dst backend/data --val_split 0.2
    os.chdir('/content/drive/MyDrive/medical-ai/backend')
else:
    print('✓ Dataset already prepared!')

## Cell 5: Verify Model Architecture

In [ ]:
# Import and verify model
from models.rickets_model import RicketsNet
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RicketsNet(num_classes=3, pretrained=True)
model = model.to(device)

print('✓ RicketsNet loaded successfully')
print(f'  Device: {device}')
print(f'  Classes: 3 (Normal, Mild_Rickets, Severe_Rickets)')
print(f'  Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Test forward pass
dummy_input = torch.randn(2, 3, 300, 300).to(device)
with torch.no_grad():
    logits, density = model(dummy_input)
    print(f'  Output shape: {logits.shape} (batch_size, num_classes)')
    print(f'  Density shape: {density.shape}')
print('\n✓ Model architecture verified!')

## Cell 6: Train Rickets Model with GPU

In [ ]:
# Train the model
import subprocess
import os

os.chdir('/content/drive/MyDrive/medical-ai/backend')

# Run training
cmd = [
    'python', 'train_rickets_cloud.py',
    '--data_dir', './data/rickets',
    '--save_dir', './checkpoints',
    '--epochs', '100',
    '--batch_size', '32',
    '--use_gpu'
]

print('Starting training on GPU...')
print('=' * 70)
result = subprocess.run(cmd, capture_output=False, text=True)
print('=' * 70)
print('Training completed!' if result.returncode == 0 else 'Training failed!')

## Cell 7: View Training Results

In [ ]:
import json
import matplotlib.pyplot as plt
import os

checkpoint_dir = './checkpoints'

# Load training history
history_path = os.path.join(checkpoint_dir, 'rickets_history.json')
if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    # Print final metrics
    print('\n' + '='*70)
    print('TRAINING RESULTS')
    print('='*70)
    print(f'Final Training Loss: {history["train_loss"][-1]:.4f}')
    print(f'Final Val Loss: {history["val_loss"][-1]:.4f}')
    print(f'Final Val Accuracy: {history["val_acc"][-1]:.2f}%')
    print(f'Final Val AUC: {history["val_auc"][-1]:.4f}')
    
    # Best metrics
    best_auc_idx = history['val_auc'].index(max(history['val_auc']))
    print(f'\nBest Val AUC: {max(history["val_auc"]):.4f} (Epoch {best_auc_idx + 1})')
    print('='*70 + '\n')
    
    # Plot curves
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2, color='green')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Validation Accuracy', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # AUC
    axes[2].plot(history['val_auc'], label='Val AUC', linewidth=2, color='orange')
    axes[2].axhline(y=max(history['val_auc']), color='red', linestyle='--', label=f'Best AUC: {max(history["val_auc"]):.4f}')
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('AUC Score', fontsize=12)
    axes[2].set_title('Validation AUC', fontsize=13, fontweight='bold')
    axes[2].set_ylim([0.8, 1.0])
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_results_display.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('✓ Training curves plotted!')
else:
    print('History file not found - training may not have completed')

## Cell 8: Download Trained Model & Results

In [ ]:
from google.colab import files
import os

checkpoint_dir = './checkpoints'

print('Files ready for download:\n')
files_to_download = [
    'rickets_best.pth',
    'rickets_history.json',
    'rickets_curves.png'
]

for filename in files_to_download:
    filepath = os.path.join(checkpoint_dir, filename)
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath) / 1e6
        print(f'  ✓ {filename} ({file_size:.1f} MB)')
    else:
        print(f'  ✗ {filename} (not found)')

print('\nDownloading...')
os.chdir(checkpoint_dir)

for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
        print(f'  ✓ Downloaded {filename}')

print('\n✓ All files downloaded to your local computer!')

## Cell 9: Next Steps

### After downloading the model:

1. **Copy the files to your local machine:**
   ```
   medical-ai/backend/checkpoints/
   ├── rickets_best.pth          (trained model)
   ├── rickets_history.json       (training metrics)
   └── rickets_curves.png         (visualization)
   ```

2. **Restart your FastAPI backend:**
   ```bash
   cd backend
   python main.py
   ```

3. **Test with your frontend:**
   - Upload X-ray images
   - Get predictions for Rickets detection

### Model Performance:
- **Expected Accuracy:** 85-92%
- **Expected AUC:** 0.95-0.98
- **Classes:** Normal / Mild_Rickets / Severe_Rickets

### To retrain with different parameters:
- Increase epochs: `--epochs 150`
- Adjust batch size: `--batch_size 64`
- Modify learning rate: `--lr 0.0005`